# NB32 · Advanced Capstone

`advanced/` · notebook 32 of 32 · 15 questions + a mini-project

This is the last notebook. Each question is a small *system*: a cache, a rate
limiter, a parser, a test runner, a job scheduler. Real programs are built
from parts like these, and each one pulls together several advanced topics
at once: classes and dataclasses (NB21), generators (NB22), decorators
(NB23), context managers (NB24), metaprogramming (NB25), type hints (NB26),
asyncio (NB27), testing (NB28) and performance thinking (NB29).

The questions start at L3. Expect to write 30 to 60 lines for the L5 ones.
Plan before you type: name the pieces, decide what state each one owns, then
build and test them one at a time.

**Determinism matters.** Anything that depends on time takes a *fake clock*
(a value you move forward yourself), randomness is seeded, and results are
sorted before printing. That is not a classroom trick: it is how real
systems code is made testable.

**Trade-offs are part of the answer.** Most of these problems have more than
one good design. The explanations and senior notes say what each choice
costs, so read them even when your output already matches.

## Prerequisites

Before you start, you should be comfortable with:

- [ ] NB21 to NB31: all advanced notebooks

## What you'll practise

- System-design-sized problems that bring the advanced topics together

## How to use this notebook

1. Read the question and write your code in the cell beneath it.
2. Run the cell, then open **Expected output** and compare.
3. Stuck? Open **Hint** first. Open **Solution** only after a real attempt.
4. Once it works, open **Senior dev solution** to see how an experienced developer would write it, and why.

The **Solution** only uses what this notebook and the ones before it teach. The **Senior dev solution** sometimes uses later topics on purpose, and says which notebook covers them.

| Type | What you do |
|---|---|
| **Write** | Write code that prints the answer |
| **Predict** | Read the code and predict exactly what it prints |
| **Error** | Predict which error the code raises |
| **Fix** | The code has a bug. Find it and fix it |
| **Refactor** | The code works. Make it cleaner without changing its output |

| Level | Meaning |
|---|---|
| L1 | Extremely Easy: one idea, a line or two |
| L2 | Easy: one idea with a small twist |
| L3 | Intermediate: two or three ideas combined |
| L4 | Somewhat Difficult: needs care to get exactly right |
| L5 | Hard: several ideas and a trap or two |

---

## L3 · Intermediate

### NB32-Q01 · L3 · Event bus with decorators · Write

Build a tiny *event bus*: parts of a program subscribe to named events,
and publishing an event calls every subscriber in the order they
subscribed.

Write a class `EventBus` with:

- a method `on(event)` that is used as a **decorator**: `@bus.on("order_paid")`
  registers the function below it and gives the function back unchanged;
- a method `publish(event, payload)` that calls each subscriber of that
  event with `payload` and returns a list of their return values (an
  empty list if nobody subscribed).

Then subscribe two handlers to `"order_paid"` (one returns
`f"email sent to {payload['email']}"`, the other returns
`f"stock reserved for {payload['sku']}"`) and print the results of
publishing `order` to `"order_paid"` and to `"order_refunded"`:

```text
['email sent to ada@example.com', 'stock reserved for KB-101']
[]
```

**Starting code (already in the cell below):**

```python
order = {"email": "ada@example.com", "sku": "KB-101"}
```

<details><summary><b>Hint</b></summary>

Store subscribers in a dictionary from event name to a list of functions. `on` must return the real decorator, which appends the function and returns it.

</details>

<details><summary><b>Expected output</b></summary>

```text
['email sent to ada@example.com', 'stock reserved for KB-101']
[]
```

</details>

<details><summary><b>Solution</b></summary>

```python
class EventBus:
    def __init__(self):
        self.handlers = {}

    def on(self, event):
        def register(func):
            if event not in self.handlers:
                self.handlers[event] = []
            self.handlers[event].append(func)
            return func
        return register

    def publish(self, event, payload):
        results = []
        for handler in self.handlers.get(event, []):
            results.append(handler(payload))
        return results


bus = EventBus()

@bus.on("order_paid")
def send_email(payload):
    return f"email sent to {payload['email']}"

@bus.on("order_paid")
def reserve_stock(payload):
    return f"stock reserved for {payload['sku']}"

print(bus.publish("order_paid", order))
print(bus.publish("order_refunded", order))
```

`@bus.on("order_paid")` runs in two steps: `bus.on("order_paid")` is called first and returns `register`, then `register(send_email)` runs. Because `register` returns the function unchanged, `send_email` is still a normal function you can call or test on its own. A decorator *with arguments* is always a function that returns a decorator (NB23). `self.handlers.get(event, [])` makes publishing to an unknown event a harmless no-op instead of a `KeyError`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from collections import defaultdict
from collections.abc import Callable
from typing import Any

type Handler = Callable[[dict[str, Any]], Any]


class EventBus:
    def __init__(self) -> None:
        self._handlers: defaultdict[str, list[Handler]] = defaultdict(list)

    def on(self, event: str) -> Callable[[Handler], Handler]:
        def register(func: Handler) -> Handler:
            self._handlers[event].append(func)
            return func
        return register

    def publish(self, event: str, payload: dict[str, Any]) -> list[Any]:
        return [handler(payload) for handler in self._handlers.get(event, ())]


bus = EventBus()

@bus.on("order_paid")
def send_email(payload):
    return f"email sent to {payload['email']}"

@bus.on("order_paid")
def reserve_stock(payload):
    return f"stock reserved for {payload['sku']}"

print(bus.publish("order_paid", order))
print(bus.publish("order_refunded", order))
```

`defaultdict(list)` removes the "create the list if missing" step, and the `type Handler = ...` alias (NB26) names the callback shape once. Publishing uses `.get(event, ())` rather than `self._handlers[event]` on purpose: indexing a `defaultdict` would *create* an empty entry for every event anyone ever publishes. The trade-off of a synchronous bus is that one slow or crashing handler blocks the rest; production buses catch errors per handler or hand events to a queue.

</details>

In [ ]:
order = {"email": "ada@example.com", "sku": "KB-101"}

# your code here


### NB32-Q02 · L3 · Lazy pipelines · Predict

A log pipeline built from generators. Each stage prints when it does
work. Predict the output, line by line.

**What does this print?**

```python
def read(lines):
    for line in lines:
        print("read", line)
        yield line

def errors_only(rows):
    for row in rows:
        if row.startswith("ERR"):
            print("keep", row)
            yield row

logs = ["ERR disk", "OK boot", "ERR net", "ERR fan"]
pipeline = errors_only(read(logs))
print("built")
print(next(pipeline))
print(next(pipeline))
```

<details><summary><b>Hint</b></summary>

Building a generator runs none of its body. Each `next()` pulls just enough items through the stages to produce one result.

</details>

<details><summary><b>Expected output</b></summary>

```text
built
read ERR disk
keep ERR disk
ERR disk
read OK boot
read ERR net
keep ERR net
ERR net
```

</details>

<details><summary><b>Solution</b></summary>

Creating `pipeline` runs no code at all, so `built` comes first. The first `next()` asks `errors_only` for a row; it asks `read`, which reads `ERR disk`, and that passes the filter. The second `next()` resumes both generators where they paused: `read` reads `OK boot` (dropped, nothing kept), then `ERR net`, which is kept. `ERR fan` is never read at all. This pull-based laziness (NB22) is why generator pipelines can process files far bigger than memory.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from itertools import islice

def read(lines):
    for line in lines:
        print("read", line)
        yield line

logs = ["ERR disk", "OK boot", "ERR net", "ERR fan"]
pipeline = (row for row in read(logs) if row.startswith("ERR"))
print(list(islice(pipeline, 2)))
```

A generator expression is the compact form of a filter stage, and `islice(pipeline, 2)` takes "the first two" without ever touching the rest of the input. The trade-off of laziness is that side effects (here, the `print` calls) happen *when items are pulled*, not when the pipeline is built. That surprises people who log in a stage and wonder why nothing appears.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB32-Q03 · L3 · Retry decorator · Write

Network calls fail now and then, so write a decorator factory
`retry(times, base_delay, sleep)` that:

- calls the wrapped function, and if it raises `ConnectionError`, tries
  again, up to `times` attempts in total;
- waits `base_delay * 2 ** (attempt - 1)` seconds between attempts
  (exponential backoff: `0.5`, `1.0`, `2.0`, ...) by calling the
  `sleep` function it was given;
- re-raises the last error if every attempt fails;
- keeps the wrapped function's `__name__`.

The given `fake_sleep` records delays instead of waiting, so the test
runs instantly. Decorate `fetch_rates` with `times=4, base_delay=0.5,
sleep=fake_sleep`, call it once, and print:

```text
fetch_rates -> rates v3
waited [0.5, 1.0]
```

**Starting code (already in the cell below):**

```python
waits = []

def fake_sleep(seconds):
    waits.append(seconds)

attempts = {"count": 0}

def fetch_rates():
    attempts["count"] += 1
    if attempts["count"] < 3:
        raise ConnectionError("timeout")
    return f"rates v{attempts['count']}"
```

<details><summary><b>Hint</b></summary>

Three layers: `retry(...)` returns a decorator, the decorator returns a `wrapper`. Inside `wrapper`, loop over attempt numbers with `try`/`except`. Only sleep if another attempt is coming. `functools.wraps` keeps the name.

</details>

<details><summary><b>Expected output</b></summary>

```text
fetch_rates -> rates v3
waited [0.5, 1.0]
```

</details>

<details><summary><b>Solution</b></summary>

```python
import functools

def retry(times, base_delay, sleep):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except ConnectionError:
                    if attempt == times:
                        raise
                    sleep(base_delay * 2 ** (attempt - 1))
        return wrapper
    return decorator

fetch_rates = retry(times=4, base_delay=0.5, sleep=fake_sleep)(fetch_rates)
print(fetch_rates.__name__, "->", fetch_rates())
print("waited", waits)
```

Attempts 1 and 2 raise, so the wrapper sleeps `0.5 * 2**0 = 0.5` and then `0.5 * 2**1 = 1.0`. Attempt 3 succeeds and `return` leaves the loop at once. A bare `raise` inside `except` re-raises the *current* error with its original traceback. Passing `sleep` in (instead of calling `time.sleep` directly) is *dependency injection*: the same decorator really waits in production and fakes it in tests (NB28).

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import functools
import time
from collections.abc import Callable

def retry[**P, R](times: int, base_delay: float,
                  sleep: Callable[[float], None] = time.sleep,
                  on: tuple[type[Exception], ...] = (ConnectionError,)):
    def decorator(func: Callable[P, R]) -> Callable[P, R]:
        @functools.wraps(func)
        def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
            for attempt in range(1, times):
                try:
                    return func(*args, **kwargs)
                except on:
                    sleep(base_delay * 2 ** (attempt - 1))
            return func(*args, **kwargs)
        return wrapper
    return decorator

fetch_rates = retry(4, 0.5, sleep=fake_sleep)(fetch_rates)
print(fetch_rates.__name__, "->", fetch_rates())
print("waited", waits)
```

The senior version lets the final attempt run *outside* the `try`, so the last error escapes naturally without an `if attempt == times` check. `ParamSpec` (`**P`, NB26) keeps the wrapped function's signature for type checkers, `time.sleep` is the real default, and `on=` makes the retried exceptions configurable. Never retry *every* exception: retrying a `ValueError` from bad input only repeats the same failure. Production code also adds random *jitter* to the delays so many clients don't retry in lock-step.

</details>

In [ ]:
waits = []

def fake_sleep(seconds):
    waits.append(seconds)

attempts = {"count": 0}

def fetch_rates():
    attempts["count"] += 1
    if attempts["count"] < 3:
        raise ConnectionError("timeout")
    return f"rates v{attempts['count']}"

# your code here


### NB32-Q04 · L3 · Transactions · Fix

`transaction(store)` should make a block *all or nothing*: if the block
raises, every change it made to `store` is undone and the error is
reported. The transfer below fails half-way (the account `"zed"`
doesn't exist), but `ana` keeps the deduction. Fix the context manager.
The correct output is:

```text
rolled back: KeyError
{'ana': 100, 'bo': 50}
```

**Buggy code (copied into the cell below):**

```python
from contextlib import contextmanager

@contextmanager
def transaction(store):
    try:
        snapshot = dict(store)
    except Exception as error:
        store.clear()
        store.update(snapshot)
        print("rolled back:", type(error).__name__)
        raise
    yield store

balances = {"ana": 100, "bo": 50}
try:
    with transaction(balances) as tx:
        tx["ana"] -= 30
        tx["zed"] += 30
except KeyError:
    pass
print(balances)
```

<details><summary><b>Hint</b></summary>

An exception raised inside the `with` block is thrown into the generator *at the `yield`*. Code after the `yield` only runs if you catch it there.

</details>

<details><summary><b>Expected output</b></summary>

```text
rolled back: KeyError
{'ana': 100, 'bo': 50}
```

</details>

<details><summary><b>Solution</b></summary>

```python
from contextlib import contextmanager

@contextmanager
def transaction(store):
    snapshot = dict(store)
    try:
        yield store
    except Exception as error:
        store.clear()
        store.update(snapshot)
        print("rolled back:", type(error).__name__)
        raise

balances = {"ana": 100, "bo": 50}
try:
    with transaction(balances) as tx:
        tx["ana"] -= 30
        tx["zed"] += 30
except KeyError:
    pass
print(balances)
```

In a `@contextmanager` generator, the `yield` is where the `with` body runs. If the body raises, the exception reappears *at the yield*. The buggy `try` guards only the snapshot line, which never fails, so the error passes straight through the generator: the rollback never runs and `ana` stays at `70`. Wrapping the `yield` in `try`/`except` gives you the place to restore the snapshot. The final `raise` matters too: a transaction should undo the damage, not hide the error from the caller (NB24).

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Transaction:
    def __init__(self, store: dict[str, int]) -> None:
        self.store = store

    def __enter__(self) -> dict[str, int]:
        self.snapshot = dict(self.store)
        return self.store

    def __exit__(self, exc_type, exc, tb) -> bool:
        if exc_type is not None:
            self.store.clear()
            self.store.update(self.snapshot)
            print("rolled back:", exc_type.__name__)
        return False

balances = {"ana": 100, "bo": 50}
try:
    with Transaction(balances) as tx:
        tx["ana"] -= 30
        tx["zed"] += 30
except KeyError:
    pass
print(balances)
```

The class form makes the protocol explicit: `__exit__` receives the exception and returning `False` lets it propagate. Both forms are fine; the class wins when the manager needs extra methods or state. The design trade-off is the snapshot itself: copying the whole store is simple and always correct but costs O(n) per transaction. Databases instead keep an *undo log* of only the keys that changed.

</details>

In [ ]:
from contextlib import contextmanager

@contextmanager
def transaction(store):
    try:
        snapshot = dict(store)
    except Exception as error:
        store.clear()
        store.update(snapshot)
        print("rolled back:", type(error).__name__)
        raise
    yield store

balances = {"ana": 100, "bo": 50}
try:
    with transaction(balances) as tx:
        tx["ana"] -= 30
        tx["zed"] += 30
except KeyError:
    pass
print(balances)


### NB32-Q05 · L3 · Immutable config · Error

Service settings are loaded once into a frozen dataclass so nothing can
change them by accident. A later line tries anyway. Which error does
this raise?

**Which error does this raise?**

```python
from dataclasses import dataclass, replace

@dataclass(frozen=True)
class Settings:
    host: str
    port: int
    debug: bool = False

base = Settings("api.local", 8080)
staging = replace(base, debug=True)
print(base.debug, staging.debug)
staging.port = 9090
```

<details><summary><b>Hint</b></summary>

`replace` builds a *new* object. Assigning to a field of a frozen one is a different story.

</details>

<details><summary><b>Expected output</b></summary>

```text
False True
FrozenInstanceError
```

</details>

<details><summary><b>Solution</b></summary>

`FrozenInstanceError` (a subclass of `AttributeError`, from the `dataclasses` module). `frozen=True` makes the generated `__setattr__` refuse every assignment. The supported way to "change" a frozen object is `dataclasses.replace`, which copies it with some fields swapped, so the first `print` works and shows `False True`. Frozen dataclasses are also hashable, so they can be dictionary keys and set members (NB21).

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import dataclass, replace

@dataclass(frozen=True, slots=True)
class Settings:
    host: str
    port: int
    debug: bool = False

base = Settings("api.local", 8080)
staging = replace(base, debug=True, port=9090)
print(base.debug, staging.debug, staging.port)
```

Configuration objects should be immutable: any part of the program can hold a reference without fear that another part changes it underneath. Changes become explicit new values via `replace`. `slots=True` (NB29) saves memory and blocks typo-attributes too. The trade-off is that "editing" means copying, which is fine for small objects that change rarely.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

---

## L4 · Somewhat Difficult

### NB32-Q06 · L4 · LRU cache with TTL · Write

Build `TTLCache(capacity, ttl, clock)`: a least-recently-used cache whose
entries also expire. `clock()` returns the current time (use the given
`FakeClock`, so time only moves when you say so).

- `put(key, value)` stores the value with the current time and makes the
  key the most recently used. If that leaves more than `capacity` keys,
  remove the **least** recently used one and count an *eviction*.
- `get(key)` returns the value and makes the key most recently used (a
  *hit*). A missing key returns `None` (a *miss*). A key whose age
  (`clock() - time it was put`) is `>= ttl` is deleted, counts as a miss
  **and** as *expired*, and returns `None`.
- `keys()` returns the keys from least to most recently used.
- `stats()` returns a dict with `hits`, `misses`, `evictions`, `expired`.

Run the given script. It must print:

```text
1
None
None
30
['c']
{'hits': 2, 'misses': 2, 'evictions': 1, 'expired': 1}
```

**Starting code (already in the cell below):**

```python
class FakeClock:
    def __init__(self):
        self.now = 0.0

    def __call__(self):
        return self.now

    def advance(self, seconds):
        self.now += seconds

# --- your TTLCache class goes here ---

# Script to run once your class exists:
# clock = FakeClock()
# cache = TTLCache(capacity=2, ttl=10, clock=clock)
# cache.put("a", 1); cache.put("b", 2)
# print(cache.get("a"))
# cache.put("c", 3)
# print(cache.get("b"))
# clock.advance(6); cache.put("c", 30)
# clock.advance(5)
# print(cache.get("a"))
# print(cache.get("c"))
# print(cache.keys())
# print(cache.stats())
```

<details><summary><b>Hint</b></summary>

`collections.OrderedDict` remembers order and has `move_to_end(key)` and `popitem(last=False)` (removes the oldest). Store `(value, stored_at)` pairs.

</details>

<details><summary><b>Expected output</b></summary>

```text
1
None
None
30
['c']
{'hits': 2, 'misses': 2, 'evictions': 1, 'expired': 1}
```

</details>

<details><summary><b>Solution</b></summary>

```python
from collections import OrderedDict

class TTLCache:
    def __init__(self, capacity, ttl, clock):
        self.capacity = capacity
        self.ttl = ttl
        self.clock = clock
        self.data = OrderedDict()
        self.counts = {"hits": 0, "misses": 0, "evictions": 0, "expired": 0}

    def put(self, key, value):
        self.data[key] = (value, self.clock())
        self.data.move_to_end(key)
        if len(self.data) > self.capacity:
            self.data.popitem(last=False)
            self.counts["evictions"] += 1

    def get(self, key):
        if key not in self.data:
            self.counts["misses"] += 1
            return None
        value, stored_at = self.data[key]
        if self.clock() - stored_at >= self.ttl:
            del self.data[key]
            self.counts["misses"] += 1
            self.counts["expired"] += 1
            return None
        self.data.move_to_end(key)
        self.counts["hits"] += 1
        return value

    def keys(self):
        return list(self.data)

    def stats(self):
        return dict(self.counts)


clock = FakeClock()
cache = TTLCache(capacity=2, ttl=10, clock=clock)
cache.put("a", 1); cache.put("b", 2)
print(cache.get("a"))
cache.put("c", 3)
print(cache.get("b"))
clock.advance(6); cache.put("c", 30)
clock.advance(5)
print(cache.get("a"))
print(cache.get("c"))
print(cache.keys())
print(cache.stats())
```

`OrderedDict` gives every operation in O(1): `move_to_end` marks a key as freshly used and `popitem(last=False)` removes the stalest one. Trace it: `get("a")` makes `a` newest, so adding `c` evicts `b`. Re-putting `c` at time 6 resets its age, so at time 11 `a` (age 11) has expired but `c` (age 5) is still fresh. Expiry is checked *lazily*, only when a key is read, which is why `a` sat in the cache until the `get`. `stats()` returns a copy so callers can't change the counters.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from collections import Counter, OrderedDict
from collections.abc import Callable, Hashable
from typing import Any

class TTLCache:
    def __init__(self, capacity: int, ttl: float, clock: Callable[[], float]) -> None:
        if capacity < 1:
            raise ValueError("capacity must be at least 1")
        self._capacity, self._ttl, self._clock = capacity, ttl, clock
        self._data: OrderedDict[Hashable, tuple[Any, float]] = OrderedDict()
        self._counts = Counter(hits=0, misses=0, evictions=0, expired=0)

    def put(self, key: Hashable, value: Any) -> None:
        self._data[key] = (value, self._clock())
        self._data.move_to_end(key)
        if len(self._data) > self._capacity:
            self._data.popitem(last=False)
            self._counts["evictions"] += 1

    def get(self, key: Hashable) -> Any | None:
        match self._data.get(key):
            case None:
                self._counts["misses"] += 1
                return None
            case (_, stored_at) if self._clock() - stored_at >= self._ttl:
                del self._data[key]
                self._counts.update(misses=1, expired=1)
                return None
            case (value, _):
                self._data.move_to_end(key)
                self._counts["hits"] += 1
                return value

    def keys(self) -> list[Hashable]:
        return list(self._data)

    def stats(self) -> dict[str, int]:
        return dict(self._counts)


clock = FakeClock()
cache = TTLCache(capacity=2, ttl=10, clock=clock)
cache.put("a", 1); cache.put("b", 2)
print(cache.get("a"))
cache.put("c", 3)
print(cache.get("b"))
clock.advance(6); cache.put("c", 30)
clock.advance(5)
print(cache.get("a"))
print(cache.get("c"))
print(cache.keys())
print(cache.stats())
```

Validation in `__init__` stops a `capacity=0` cache that silently stores nothing, and `match` with a guard (NB18) reads as the three cases. The big trade-off is *lazy* versus *active* expiry: lazy costs nothing until a key is read, but dead entries can occupy memory (and count towards capacity) for a long time. Caches like Redis do both: lazy checks plus a periodic sweep of a few random keys. A second trade-off: returning `None` for a miss is ambiguous if `None` can be a cached value; a sentinel object fixes that.

</details>

In [ ]:
class FakeClock:
    def __init__(self):
        self.now = 0.0

    def __call__(self):
        return self.now

    def advance(self, seconds):
        self.now += seconds

# --- your TTLCache class goes here ---

# Script to run once your class exists:
# clock = FakeClock()
# cache = TTLCache(capacity=2, ttl=10, clock=clock)
# cache.put("a", 1); cache.put("b", 2)
# print(cache.get("a"))
# cache.put("c", 3)
# print(cache.get("b"))
# clock.advance(6); cache.put("c", 30)
# clock.advance(5)
# print(cache.get("a"))
# print(cache.get("c"))
# print(cache.keys())
# print(cache.stats())

# your code here


### NB32-Q07 · L4 · Token-bucket rate limiter · Write

An API allows each client a short burst of requests, then a steady rate.
Implement the **token bucket** algorithm:

- Each client has a bucket holding up to `capacity` tokens. A new client
  starts with a full bucket.
- Tokens refill continuously at `rate` per second, never above
  `capacity`. Refill lazily: when a request arrives, add
  `(now - last_seen) * rate` tokens, then remember `now`.
- A request is allowed if the bucket has at least 1 token, which it then
  spends. Otherwise it is limited.

Write `RateLimiter(capacity, rate, clock)` with `allow(client)` returning
`True` or `False`. Using `capacity=3, rate=1`, process `requests` (each is
`(time, client)`: set `clock.now = time` first), print one line per
request, then print how many requests each client had allowed:

```text
0.0 ana ok
0.0 ana ok
0.0 ana ok
0.0 ana limited
0.0 bo ok
0.5 ana limited
1.0 ana ok
1.0 bo ok
2.5 ana ok
2.5 ana limited
{'ana': 5, 'bo': 2}
```

**Starting code (already in the cell below):**

```python
class FakeClock:
    def __init__(self):
        self.now = 0.0

    def __call__(self):
        return self.now

    def advance(self, seconds):
        self.now += seconds

requests = [(0.0, "ana"), (0.0, "ana"), (0.0, "ana"), (0.0, "ana"), (0.0, "bo"),
            (0.5, "ana"), (1.0, "ana"), (1.0, "bo"), (2.5, "ana"), (2.5, "ana")]
```

<details><summary><b>Hint</b></summary>

Keep one small record per client: tokens and the time you last refilled it. `min(capacity, ...)` caps the refill. Count allowed requests in a dict as you go.

</details>

<details><summary><b>Expected output</b></summary>

```text
0.0 ana ok
0.0 ana ok
0.0 ana ok
0.0 ana limited
0.0 bo ok
0.5 ana limited
1.0 ana ok
1.0 bo ok
2.5 ana ok
2.5 ana limited
{'ana': 5, 'bo': 2}
```

</details>

<details><summary><b>Solution</b></summary>

```python
class RateLimiter:
    def __init__(self, capacity, rate, clock):
        self.capacity = capacity
        self.rate = rate
        self.clock = clock
        self.buckets = {}

    def allow(self, client):
        now = self.clock()
        if client not in self.buckets:
            self.buckets[client] = [self.capacity, now]
        tokens, last = self.buckets[client]
        tokens = min(self.capacity, tokens + (now - last) * self.rate)
        allowed = tokens >= 1
        if allowed:
            tokens -= 1
        self.buckets[client] = [tokens, now]
        return allowed


clock = FakeClock()
limiter = RateLimiter(capacity=3, rate=1, clock=clock)
allowed_counts = {}
for time, client in requests:
    clock.now = time
    ok = limiter.allow(client)
    print(time, client, "ok" if ok else "limited")
    if ok:
        allowed_counts[client] = allowed_counts.get(client, 0) + 1
print(allowed_counts)
```

The bucket never needs a timer: refilling *on demand* from the elapsed time gives the same answer as refilling every instant. At time 0 `ana` spends all 3 tokens, so the 4th request fails. By 0.5 she has only 0.5 tokens; by 1.0 she has 1 again. At 2.5 she has 1.5: one request passes and 0.5 remains, which is not enough for the next. `bo` has his own bucket, so `ana`'s burst never affects him.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from collections import Counter
from collections.abc import Callable
from dataclasses import dataclass

@dataclass(slots=True)
class Bucket:
    tokens: float
    last: float

class RateLimiter:
    def __init__(self, capacity: float, rate: float, clock: Callable[[], float]) -> None:
        self.capacity, self.rate, self.clock = capacity, rate, clock
        self._buckets: dict[str, Bucket] = {}

    def allow(self, client: str, cost: float = 1) -> bool:
        now = self.clock()
        bucket = self._buckets.setdefault(client, Bucket(self.capacity, now))
        bucket.tokens = min(self.capacity, bucket.tokens + (now - bucket.last) * self.rate)
        bucket.last = now
        if bucket.tokens < cost:
            return False
        bucket.tokens -= cost
        return True


clock = FakeClock()
limiter = RateLimiter(capacity=3, rate=1, clock=clock)
allowed = Counter()
for clock.now, client in requests:
    ok = limiter.allow(client)
    print(clock.now, client, "ok" if ok else "limited")
    allowed[client] += ok
print(dict(allowed))
```

A small `@dataclass(slots=True)` (NB21, NB29) replaces the anonymous `[tokens, last]` list, and `cost=` lets expensive endpoints spend more than one token. `for clock.now, client in requests` assigns straight into the attribute: legal, compact, but use it sparingly. Trade-offs: a token bucket allows bursts (good for humans clicking around); a *leaky bucket* or fixed window smooths traffic but punishes bursts. Also, `_buckets` grows forever: real limiters expire idle clients, for example with the TTL cache from Q06.

</details>

In [ ]:
class FakeClock:
    def __init__(self):
        self.now = 0.0

    def __call__(self):
        return self.now

    def advance(self, seconds):
        self.now += seconds

requests = [(0.0, "ana"), (0.0, "ana"), (0.0, "ana"), (0.0, "ana"), (0.0, "bo"),
            (0.5, "ana"), (1.0, "ana"), (1.0, "bo"), (2.5, "ana"), (2.5, "ana")]

# your code here


### NB32-Q08 · L4 · functools.lru_cache · Predict

A price lookup is cached with a tiny `lru_cache`. Predict which lookups
really run and what `cache_info()` reports.

**What does this print?**

```python
from functools import lru_cache

@lru_cache(maxsize=2)
def price(sku):
    print("lookup", sku)
    return len(sku) * 10

for sku in ["A1", "B22", "A1", "C333", "B22", "A1"]:
    price(sku)
print(price.cache_info())
```

<details><summary><b>Hint</b></summary>

Track the two cached keys after every call, newest last. A hit moves a key to the newest position; a miss with a full cache drops the oldest.

</details>

<details><summary><b>Expected output</b></summary>

```text
lookup A1
lookup B22
lookup C333
lookup B22
lookup A1
CacheInfo(hits=1, misses=5, maxsize=2, currsize=2)
```

</details>

<details><summary><b>Solution</b></summary>

`A1` and `B22` miss and fill the cache. `A1` hits and becomes newest, so the cache is `[B22, A1]`. `C333` misses and evicts `B22`. Now `B22` misses again and evicts `A1`, and the final `A1` misses and evicts `C333`. That is 1 hit and 5 misses: five lookups for three distinct keys. When the working set (3 keys) is bigger than the cache (2), LRU can *thrash*, evicting exactly the key you need next.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import lru_cache

@lru_cache(maxsize=4)
def price(sku):
    print("lookup", sku)
    return len(sku) * 10

for sku in ["A1", "B22", "A1", "C333", "B22", "A1"]:
    price(sku)
print(price.cache_info())
```

Seniors size a cache from measurement: `cache_info()` is the tool. A low hit rate with a full cache means the working set doesn't fit, and here a `maxsize` of 4 turns 5 misses into 3. The trade-off is memory, and `lru_cache` keeps strong references to arguments and results. Never put it on methods of short-lived objects: the cache keeps `self` alive (NB29).

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB32-Q09 · L4 · State machines · Refactor

This order workflow works, but every new state or action means more
nested branches. Refactor it so the allowed transitions are **data**
rather than code. The output must stay the same.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
def advance(status, action):
    if status == "new":
        if action == "pay":
            return "paid"
        elif action == "cancel":
            return "cancelled"
        else:
            return None
    elif status == "paid":
        if action == "ship":
            return "shipped"
        elif action == "refund":
            return "refunded"
        else:
            return None
    elif status == "shipped":
        if action == "deliver":
            return "delivered"
        else:
            return None
    else:
        return None

status = "new"
for action in ["pay", "deliver", "ship", "deliver", "refund"]:
    result = advance(status, action)
    if result is None:
        print(f"{status}: cannot {action}")
    else:
        print(f"{status} -> {result}")
        status = result
```

<details><summary><b>Hint</b></summary>

A dictionary keyed by `(status, action)` pairs can hold every arrow of the workflow diagram. A missing key means the move is not allowed.

</details>

<details><summary><b>Expected output</b></summary>

```text
new -> paid
paid: cannot deliver
paid -> shipped
shipped -> delivered
delivered: cannot refund
```

</details>

<details><summary><b>Solution</b></summary>

```python
TRANSITIONS = {
    ("new", "pay"): "paid",
    ("new", "cancel"): "cancelled",
    ("paid", "ship"): "shipped",
    ("paid", "refund"): "refunded",
    ("shipped", "deliver"): "delivered",
}

status = "new"
for action in ["pay", "deliver", "ship", "deliver", "refund"]:
    result = TRANSITIONS.get((status, action))
    if result is None:
        print(f"{status}: cannot {action}")
    else:
        print(f"{status} -> {result}")
        status = result
```

Every branch of the original function was really one arrow in a diagram: from a state, an action leads to a new state. A dictionary keyed by `(status, action)` tuples stores those arrows directly, and `.get()` returns `None` for any arrow that doesn't exist. Adding a state is now one line of data, and you can print, test or draw the table itself.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from enum import StrEnum

class Status(StrEnum):
    NEW = "new"
    PAID = "paid"
    SHIPPED = "shipped"
    DELIVERED = "delivered"
    CANCELLED = "cancelled"
    REFUNDED = "refunded"

class InvalidTransition(Exception):
    pass

TRANSITIONS: dict[Status, dict[str, Status]] = {
    Status.NEW: {"pay": Status.PAID, "cancel": Status.CANCELLED},
    Status.PAID: {"ship": Status.SHIPPED, "refund": Status.REFUNDED},
    Status.SHIPPED: {"deliver": Status.DELIVERED},
}

def advance(status: Status, action: str) -> Status:
    try:
        return TRANSITIONS[status][action]
    except KeyError:
        raise InvalidTransition(f"{status}: cannot {action}") from None

status = Status.NEW
for action in ["pay", "deliver", "ship", "deliver", "refund"]:
    try:
        new_status = advance(status, action)
    except InvalidTransition as error:
        print(error)
    else:
        print(f"{status} -> {new_status}")
        status = new_status
```

A `StrEnum` (NB21) turns typos like `"shiped"` into immediate errors while still printing as plain text. Raising a named `InvalidTransition` instead of returning `None` means a caller can't forget to check. The nested `{state: {action: state}}` shape also answers "what can I do from here?" with `TRANSITIONS[status].keys()`, handy for building UI buttons. Trade-off: tables are great until transitions need conditions ("refund only within 30 days"); then each arrow carries a guard function too.

</details>

In [ ]:
def advance(status, action):
    if status == "new":
        if action == "pay":
            return "paid"
        elif action == "cancel":
            return "cancelled"
        else:
            return None
    elif status == "paid":
        if action == "ship":
            return "shipped"
        elif action == "refund":
            return "refunded"
        else:
            return None
    elif status == "shipped":
        if action == "deliver":
            return "delivered"
        else:
            return None
    else:
        return None

status = "new"
for action in ["pay", "deliver", "ship", "deliver", "refund"]:
    result = advance(status, action)
    if result is None:
        print(f"{status}: cannot {action}")
    else:
        print(f"{status} -> {result}")
        status = result


### NB32-Q10 · L4 · Plugin registry · Write

Build an exporter system where **defining a subclass is enough to
register it**. Write a base class `Exporter` that uses
`__init_subclass__(cls, format, **kwargs)` to record each subclass in
`Exporter.registry` under its `format` keyword. Then write three
subclasses, each with an `export(record)` method:

- `format="csv"`: values joined with commas, in the record's order;
- `format="json"`: `json.dumps(record, sort_keys=True)`;
- `format="kv"`: `key=value` pairs joined with spaces, in the record's order.

Write a function `export(record, fmt)` that looks up the class, creates
one and returns its output, or raises `ValueError` with the message
shown below for an unknown format. Print every format in sorted order,
then try `"xml"` and print the error message:

```text
csv: 7,Desk lamp,24.5
json: {"id": 7, "name": "Desk lamp", "price": 24.5}
kv: id=7 name=Desk lamp price=24.5
unknown format 'xml' (choose from csv, json, kv)
```

**Starting code (already in the cell below):**

```python
import json

record = {"id": 7, "name": "Desk lamp", "price": 24.5}
```

<details><summary><b>Hint</b></summary>

`class CsvExporter(Exporter, format="csv"):` passes `format` to `__init_subclass__`. Call `super().__init_subclass__(**kwargs)` too. `{fmt!r}` puts quotes around the name in an f-string.

</details>

<details><summary><b>Expected output</b></summary>

```text
csv: 7,Desk lamp,24.5
json: {"id": 7, "name": "Desk lamp", "price": 24.5}
kv: id=7 name=Desk lamp price=24.5
unknown format 'xml' (choose from csv, json, kv)
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Exporter:
    registry = {}

    def __init_subclass__(cls, format, **kwargs):
        super().__init_subclass__(**kwargs)
        Exporter.registry[format] = cls

class CsvExporter(Exporter, format="csv"):
    def export(self, record):
        return ",".join(str(value) for value in record.values())

class JsonExporter(Exporter, format="json"):
    def export(self, record):
        return json.dumps(record, sort_keys=True)

class KvExporter(Exporter, format="kv"):
    def export(self, record):
        return " ".join(f"{key}={value}" for key, value in record.items())

def export(record, fmt):
    if fmt not in Exporter.registry:
        choices = ", ".join(sorted(Exporter.registry))
        raise ValueError(f"unknown format {fmt!r} (choose from {choices})")
    return Exporter.registry[fmt]().export(record)

for fmt in sorted(Exporter.registry):
    print(f"{fmt}: {export(record, fmt)}")
try:
    export(record, "xml")
except ValueError as error:
    print(error)
```

`__init_subclass__` runs once for every class that inherits from `Exporter`, at the moment the `class` statement finishes (NB25). Keyword arguments in the class header, like `format="csv"`, arrive as its parameters. So the registry fills itself: nobody has to remember to add a new exporter to a list. The error message lists the real choices, built from the registry, so it can never go out of date.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import json
from abc import ABC, abstractmethod
from collections.abc import Mapping
from typing import Any, ClassVar

class Exporter(ABC):
    registry: ClassVar[dict[str, type["Exporter"]]] = {}

    def __init_subclass__(cls, *, format: str, **kwargs: Any) -> None:
        super().__init_subclass__(**kwargs)
        if format in Exporter.registry:
            raise TypeError(f"format {format!r} already registered")
        Exporter.registry[format] = cls

    @abstractmethod
    def export(self, record: Mapping[str, Any]) -> str: ...

class CsvExporter(Exporter, format="csv"):
    def export(self, record):
        return ",".join(map(str, record.values()))

class JsonExporter(Exporter, format="json"):
    def export(self, record):
        return json.dumps(record, sort_keys=True)

class KvExporter(Exporter, format="kv"):
    def export(self, record):
        return " ".join(f"{key}={value}" for key, value in record.items())

def export(record: Mapping[str, Any], fmt: str) -> str:
    try:
        exporter_cls = Exporter.registry[fmt]
    except KeyError:
        choices = ", ".join(sorted(Exporter.registry))
        raise ValueError(f"unknown format {fmt!r} (choose from {choices})") from None
    return exporter_cls().export(record)

for fmt in sorted(Exporter.registry):
    print(f"{fmt}: {export(record, fmt)}")
try:
    export(record, "xml")
except ValueError as error:
    print(error)
```

`ABC` plus `@abstractmethod` (NB21) means a subclass that forgets `export` can't even be created, and the duplicate check stops two plugins silently fighting over one name. `from None` hides the internal `KeyError` from the traceback. The trade-off of self-registering classes is *import-time magic*: a plugin only exists if its module has been imported, which is why real frameworks pair this with entry points or an explicit import list.

</details>

In [ ]:
import json

record = {"id": 7, "name": "Desk lamp", "price": 24.5}

# your code here


---

## L5 · Hard

### NB32-Q11 · L5 · Cooperative mixins · Predict

A storage class is assembled from two mixins that each wrap `save`.
Predict the output, including the last line.

**What does this print?**

```python
class Store:
    def save(self, item):
        print("store", item)

class LoggingMixin:
    def save(self, item):
        print("log start")
        super().save(item)
        print("log end")

class ValidationMixin:
    def save(self, item):
        print("validate")
        super().save(item)

class SafeStore(LoggingMixin, ValidationMixin, Store):
    pass

SafeStore().save("invoice-9")
print([cls.__name__ for cls in SafeStore.__mro__])
```

<details><summary><b>Hint</b></summary>

`super()` does not mean "my parent class". It means "the next class in the MRO of the object's actual type". Write that MRO out first.

</details>

<details><summary><b>Expected output</b></summary>

```text
log start
validate
store invoice-9
log end
['SafeStore', 'LoggingMixin', 'ValidationMixin', 'Store', 'object']
```

</details>

<details><summary><b>Solution</b></summary>

The MRO of `SafeStore` is `SafeStore, LoggingMixin, ValidationMixin, Store, object`. `LoggingMixin` has no base class besides `object`, yet its `super().save` reaches `ValidationMixin`, because `super()` follows the MRO of `SafeStore`, the real type of `self` (NB21). So the calls nest like layers of an onion: log start, validate, store, and only then log end. Swapping the mixins in the class header would log *after* validating starts.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Store:
    def save(self, item: str) -> None:
        print("store", item)

class LoggingMixin(Store):
    def save(self, item: str) -> None:
        print("log start")
        try:
            super().save(item)
        finally:
            print("log end")

class ValidationMixin(Store):
    def save(self, item: str) -> None:
        print("validate")
        super().save(item)

class SafeStore(LoggingMixin, ValidationMixin):
    pass

SafeStore().save("invoice-9")
print([cls.__name__ for cls in SafeStore.__mro__])
```

Making each mixin inherit from `Store` documents what it wraps and lets type checkers see that `super().save` exists, and the MRO stays the same. `try`/`finally` guarantees "log end" even when saving fails. The trade-off of mixin stacks: behaviour depends on the *order* of bases, which is invisible at the call site. Many teams prefer composition (a `SafeStore` that *holds* a store and calls it) once there are more than two layers.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB32-Q12 · L5 · Expression language · Write

Build a calculator language in three stages, the same way real
interpreters work.

1. **Tokenizer**: `tokenize(text)` turns `"2 * (x + 3)"` into a list of
   tokens: whole numbers (as `int`), names (letters only) and the
   symbols `+ - * / ( )`. Spaces are skipped. Any other character
   raises `SyntaxError`.
2. **Parser**: builds a tree from frozen dataclasses `Num(value)`,
   `Var(name)` and `BinOp(op, left, right)`. `*` and `/` bind tighter
   than `+` and `-`, brackets override that, and operators of equal
   strength group **from the left**: `10 - 3 - 2` means `(10 - 3) - 2`.
3. **Evaluator**: `evaluate(node, env)` computes the value, looking
   names up in the dict `env`. `/` is true division.

Print the tree for `"2 * (x + 3)"`, then evaluate each of `exprs` with
`env`, printing `text = value`. For `"y + 1"` (an unknown name) print
`y + 1 = error: unknown name 'y'`. Then print what the tokenizer says
about `"2 $ 3"`:

```text
BinOp(op='*', left=Num(value=2), right=BinOp(op='+', left=Var(name='x'), right=Num(value=3)))
2 * (x + 3) = 16
10 - 3 - 2 = 5
rate * hours + bonus / 2 = 360.0
y + 1 = error: unknown name 'y'
SyntaxError: unexpected character '$'
```

This is a *recursive descent* parser. One function per precedence
level works well:

```python
# expr   := term (("+" | "-") term)*
# term   := factor (("*" | "/") factor)*
# factor := NUMBER | NAME | "(" expr ")"
```

**Starting code (already in the cell below):**

```python
exprs = ["2 * (x + 3)", "10 - 3 - 2", "rate * hours + bonus / 2", "y + 1"]
env = {"x": 5, "rate": 40, "hours": 8, "bonus": 80}
```

<details><summary><b>Hint</b></summary>

Tokenize with an index and a `while` loop, collecting runs of digits or letters. For the parser, keep a position into the token list and write `expr`, `term` and `factor` functions that call each other; a `while` loop inside `expr` and `term` gives left-to-right grouping. The evaluator is a `match` on the node type.

</details>

<details><summary><b>Expected output</b></summary>

```text
BinOp(op='*', left=Num(value=2), right=BinOp(op='+', left=Var(name='x'), right=Num(value=3)))
2 * (x + 3) = 16
10 - 3 - 2 = 5
rate * hours + bonus / 2 = 360.0
y + 1 = error: unknown name 'y'
SyntaxError: unexpected character '$'
```

</details>

<details><summary><b>Solution</b></summary>

```python
from dataclasses import dataclass

@dataclass(frozen=True)
class Num:
    value: int

@dataclass(frozen=True)
class Var:
    name: str

@dataclass(frozen=True)
class BinOp:
    op: str
    left: object
    right: object


def tokenize(text):
    tokens = []
    i = 0
    while i < len(text):
        ch = text[i]
        if ch.isspace():
            i += 1
        elif ch.isdigit():
            start = i
            while i < len(text) and text[i].isdigit():
                i += 1
            tokens.append(int(text[start:i]))
        elif ch.isalpha():
            start = i
            while i < len(text) and text[i].isalpha():
                i += 1
            tokens.append(text[start:i])
        elif ch in "+-*/()":
            tokens.append(ch)
            i += 1
        else:
            raise SyntaxError(f"unexpected character {ch!r}")
    return tokens


class Parser:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0

    def peek(self):
        if self.pos < len(self.tokens):
            return self.tokens[self.pos]
        return None

    def take(self):
        token = self.peek()
        self.pos += 1
        return token

    def expr(self):
        node = self.term()
        while self.peek() in ("+", "-"):
            op = self.take()
            node = BinOp(op, node, self.term())
        return node

    def term(self):
        node = self.factor()
        while self.peek() in ("*", "/"):
            op = self.take()
            node = BinOp(op, node, self.factor())
        return node

    def factor(self):
        token = self.take()
        if isinstance(token, int):
            return Num(token)
        if token == "(":
            node = self.expr()
            if self.take() != ")":
                raise SyntaxError("expected ')'")
            return node
        if isinstance(token, str) and token.isalpha():
            return Var(token)
        raise SyntaxError(f"unexpected token {token!r}")


def parse(text):
    parser = Parser(tokenize(text))
    node = parser.expr()
    if parser.peek() is not None:
        raise SyntaxError(f"unexpected token {parser.peek()!r}")
    return node


def evaluate(node, env):
    match node:
        case Num(value):
            return value
        case Var(name):
            if name not in env:
                raise NameError(f"unknown name {name!r}")
            return env[name]
        case BinOp(op, left, right):
            a = evaluate(left, env)
            b = evaluate(right, env)
            if op == "+":
                return a + b
            if op == "-":
                return a - b
            if op == "*":
                return a * b
            return a / b


print(parse("2 * (x + 3)"))
for text in exprs:
    try:
        print(text, "=", evaluate(parse(text), env))
    except NameError as error:
        print(text, "= error:", error)
try:
    tokenize("2 $ 3")
except SyntaxError as error:
    print("SyntaxError:", error)
```

Each stage has one job. The tokenizer deals with characters, so the parser only ever sees clean tokens. The parser's function-per-level shape *is* the precedence: `expr` can only combine whole `term`s, so `*` is always grouped first. The `while` loops build the tree leftwards (`BinOp('-', BinOp('-', 10, 3), 2)`), which is the trap: a version that recurses on the right (`expr := term '-' expr`) quietly computes `10 - (3 - 2) = 9`. The evaluator walks the tree with `match` and class patterns (NB18): dataclasses support positional patterns like `Num(value)` automatically. Frozen dataclasses also give a readable `repr` for free, which is why the tree prints so clearly.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import operator
import re
from dataclasses import dataclass

@dataclass(frozen=True, slots=True)
class Num:
    value: int

@dataclass(frozen=True, slots=True)
class Var:
    name: str

@dataclass(frozen=True, slots=True)
class BinOp:
    op: str
    left: "Node"
    right: "Node"

type Node = Num | Var | BinOp

TOKEN_RE = re.compile(r"\s*(?:(\d+)|([A-Za-z]+)|([-+*/()])|(\S))")
OPS = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}
LEVELS = [("+", "-"), ("*", "/")]


def tokenize(text: str) -> list[int | str]:
    tokens: list[int | str] = []
    for number, name, symbol, bad in TOKEN_RE.findall(text):
        if bad:
            raise SyntaxError(f"unexpected character {bad!r}")
        tokens.append(int(number) if number else name or symbol)
    return tokens


def parse(text: str) -> Node:
    tokens = tokenize(text)
    pos = 0

    def binary(level: int) -> Node:
        nonlocal pos
        if level == len(LEVELS):
            return atom()
        node = binary(level + 1)
        while pos < len(tokens) and tokens[pos] in LEVELS[level]:
            op = tokens[pos]
            pos += 1
            node = BinOp(op, node, binary(level + 1))
        return node

    def atom() -> Node:
        nonlocal pos
        token = tokens[pos] if pos < len(tokens) else None
        pos += 1
        match token:
            case int():
                return Num(token)
            case "(":
                node = binary(0)
                if pos >= len(tokens) or tokens[pos] != ")":
                    raise SyntaxError("expected ')'")
                pos += 1
                return node
            case str() if token.isalpha():
                return Var(token)
        raise SyntaxError(f"unexpected token {token!r}")

    node = binary(0)
    if pos != len(tokens):
        raise SyntaxError(f"unexpected token {tokens[pos]!r}")
    return node


def evaluate(node: Node, env: dict[str, float]) -> float:
    match node:
        case Num(value):
            return value
        case Var(name) if name in env:
            return env[name]
        case Var(name):
            raise NameError(f"unknown name {name!r}")
        case BinOp(op, left, right):
            return OPS[op](evaluate(left, env), evaluate(right, env))


print(parse("2 * (x + 3)"))
for text in exprs:
    try:
        print(text, "=", evaluate(parse(text), env))
    except NameError as error:
        print(text, "= error:", error)
try:
    tokenize("2 $ 3")
except SyntaxError as error:
    print("SyntaxError:", error)
```

A regex with one group per token kind (NB17) replaces the hand-written scanner, and a catch-all `(\S)` group turns any stray character into a clear error. The `LEVELS` table makes precedence data: adding `%` or `**` is a one-line change instead of a new method (though `**` groups from the right and would need its own rule). `operator.add` and friends replace the `if` chain. The design trade-off is *never* to use `eval()` for this: it is shorter, but it runs any Python the user types, so a calculator becomes a remote shell.

</details>

In [ ]:
exprs = ["2 * (x + 3)", "10 - 3 - 2", "rate * hours + bonus / 2", "y + 1"]
env = {"x": 5, "rate": 40, "hours": 8, "bonus": 80}

# your code here


### NB32-Q13 · L5 · Mini test framework · Write

Write a tiny version of pytest. Test functions live in the dict
`suite` (name to function), the way a module's namespace would.

- **Discovery**: run every function whose name starts with `test_`, in
  sorted name order. Ignore everything else.
- **Fixtures**: functions registered with the given `@fixture` decorator
  provide values. If a test has a parameter called `cart`, call the
  fixture named `cart` and pass its result in. Make a **fresh** value
  for every test, so tests can't leak state into each other. Use
  `inspect.signature(func).parameters` to read parameter names.
- **Reporting**: a test *passes* if it returns normally, *fails* on
  `AssertionError` and is an *error* on any other exception. Print one
  line per test, then a summary:

```text
PASS  test_add_item
FAIL  test_discount: expected 90, got 100
PASS  test_empty_total
ERROR test_missing_price: KeyError
PASS  test_starts_empty
3 passed, 1 failed, 1 errors
```

Print the `AssertionError`'s message after a failure, and only the
exception **class name** after an error.

**Starting code (already in the cell below):**

```python
FIXTURES = {}

def fixture(func):
    FIXTURES[func.__name__] = func
    return func

@fixture
def prices():
    return {"pen": 2, "book": 12}

@fixture
def cart():
    return []

def test_add_item(cart):
    cart.append("pen")
    assert cart == ["pen"]

def test_starts_empty(cart):
    assert cart == [], "cart leaked between tests"

def test_discount(prices):
    total = prices["book"] * 10 - 20
    assert total == 90, f"expected 90, got {total}"

def test_empty_total():
    assert sum([]) == 0

def test_missing_price(prices, cart):
    cart.append("lamp")
    return prices[cart[0]]

def helper_not_a_test():
    raise RuntimeError("should never run")

suite = {name: obj for name, obj in list(globals().items()) if callable(obj)}
```

<details><summary><b>Hint</b></summary>

Loop over `sorted(suite)`. For each test, build a `kwargs` dict by calling `FIXTURES[name]()` for every parameter name, then call `func(**kwargs)` inside `try` with two `except` clauses, most specific first.

</details>

<details><summary><b>Expected output</b></summary>

```text
PASS  test_add_item
FAIL  test_discount: expected 90, got 100
PASS  test_empty_total
ERROR test_missing_price: KeyError
PASS  test_starts_empty
3 passed, 1 failed, 1 errors
```

</details>

<details><summary><b>Solution</b></summary>

```python
import inspect

def run_tests(suite):
    counts = {"passed": 0, "failed": 0, "errors": 0}
    for name in sorted(suite):
        if not name.startswith("test_"):
            continue
        func = suite[name]
        kwargs = {}
        for param in inspect.signature(func).parameters:
            kwargs[param] = FIXTURES[param]()
        try:
            func(**kwargs)
        except AssertionError as error:
            counts["failed"] += 1
            print(f"FAIL  {name}: {error}")
        except Exception as error:
            counts["errors"] += 1
            print(f"ERROR {name}: {type(error).__name__}")
        else:
            counts["passed"] += 1
            print(f"PASS  {name}")
    print(f"{counts['passed']} passed, {counts['failed']} failed, {counts['errors']} errors")

run_tests(suite)
```

Discovery is a naming convention plus a filter, exactly as in pytest. `inspect.signature` reads the parameter names at runtime, so a test *declares* what it needs and the runner supplies it: that is dependency injection again. Calling the fixture once per test is why `test_starts_empty` passes even though `test_add_item` appended to its cart. The `except AssertionError` must come before `except Exception`, because `AssertionError` *is* an `Exception` and the first matching clause wins. `else` runs only when nothing was raised, which keeps the "pass" path separate from the `try`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import inspect
from collections import Counter
from collections.abc import Callable
from dataclasses import dataclass
from typing import Any, Literal

type Outcome = Literal["passed", "failed", "errors"]

@dataclass(frozen=True, slots=True)
class Result:
    name: str
    outcome: Outcome
    detail: str = ""

def run_one(name: str, func: Callable[..., Any]) -> Result:
    try:
        kwargs = {p: FIXTURES[p]() for p in inspect.signature(func).parameters}
        func(**kwargs)
    except AssertionError as error:
        return Result(name, "failed", str(error))
    except Exception as error:
        return Result(name, "errors", type(error).__name__)
    return Result(name, "passed")

def run_tests(suite: dict[str, Callable[..., Any]]) -> list[Result]:
    return [run_one(name, suite[name]) for name in sorted(suite) if name.startswith("test_")]

LABELS = {"passed": "PASS ", "failed": "FAIL ", "errors": "ERROR"}
results = run_tests(suite)
for r in results:
    print(f"{LABELS[r.outcome]} {r.name}" + (f": {r.detail}" if r.detail else ""))
counts = Counter(r.outcome for r in results)
print(f"{counts['passed']} passed, {counts['failed']} failed, {counts['errors']} errors")
```

Seniors separate *running* from *reporting*: `run_tests` returns `Result` objects, and printing is a separate step, so the same results could feed a JUnit XML file or a CI dashboard. Fixture lookup also moved inside the `try`, so a broken or missing fixture is reported as an error for that one test instead of crashing the whole run. Real pytest adds fixture *scopes* (per test, module or session): a shared database fixture is faster but risks exactly the leaking state that per-test fixtures prevent. That is the trade-off.

</details>

In [ ]:
FIXTURES = {}

def fixture(func):
    FIXTURES[func.__name__] = func
    return func

@fixture
def prices():
    return {"pen": 2, "book": 12}

@fixture
def cart():
    return []

def test_add_item(cart):
    cart.append("pen")
    assert cart == ["pen"]

def test_starts_empty(cart):
    assert cart == [], "cart leaked between tests"

def test_discount(prices):
    total = prices["book"] * 10 - 20
    assert total == 90, f"expected 90, got {total}"

def test_empty_total():
    assert sum([]) == 0

def test_missing_price(prices, cart):
    cart.append("lamp")
    return prices[cart[0]]

def helper_not_a_test():
    raise RuntimeError("should never run")

suite = {name: obj for name, obj in list(globals().items()) if callable(obj)}

# your code here


### NB32-Q14 · L5 · Async job scheduler · Write

Write an asyncio job scheduler with priorities, retries and
cancellation. Run it with top-level `await` (Jupyter allows it).

- `Scheduler` holds an `asyncio.PriorityQueue` of
  `(priority, seq, name)` entries. Lower priority numbers run first.
  `seq` is a counter that goes up by one on **every** put, so equal
  priorities run first-in, first-out (and names are never compared).
- `submit(name, priority, job)` stores the job (an `async` function
  taking the attempt number) and queues it.
- `cancel(name)` marks a job as cancelled. When a cancelled job comes out
  of the queue, print `cancelled <name>` and skip it.
- `run(max_attempts)` starts **one** worker task and waits until the
  queue is fully processed (`queue.join()`), then cancels the worker.
  The worker runs each job: on success print `done <name>: <result>`;
  on `ConnectionError` print `retry <name> (attempt <n>)` and queue it
  again with the **same priority** and a new `seq`, unless it has
  already used `max_attempts`, in which case print `failed <name>`.

Submit the jobs as shown in the given code, cancel `"cleanup"`, then
`await scheduler.run(max_attempts=2)`. Expected output:

```text
done backup: ok on attempt 1
retry email (attempt 1)
cancelled cleanup
done email: ok on attempt 2
retry sync (attempt 1)
failed sync
done report: ok on attempt 1
```

**Starting code (already in the cell below):**

```python
import asyncio

def flaky(fail_times):
    async def job(attempt):
        await asyncio.sleep(0)
        if attempt <= fail_times:
            raise ConnectionError("network down")
        return f"ok on attempt {attempt}"
    return job

# submit these in this order, then cancel "cleanup":
jobs = [("report", 5, flaky(0)), ("backup", 1, flaky(0)), ("email", 2, flaky(1)),
        ("cleanup", 2, flaky(0)), ("sync", 3, flaky(5))]
```

<details><summary><b>Hint</b></summary>

Keep `jobs`, `attempts` and `cancelled` as dictionaries/sets on the scheduler. The worker is an endless `while True` loop around `await queue.get()`; call `queue.task_done()` in a `finally` so `join()` can finish. A retry `put` happens *before* `task_done`, so the queue never looks empty too early.

</details>

<details><summary><b>Expected output</b></summary>

```text
done backup: ok on attempt 1
retry email (attempt 1)
cancelled cleanup
done email: ok on attempt 2
retry sync (attempt 1)
failed sync
done report: ok on attempt 1
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Scheduler:
    def __init__(self):
        self.queue = asyncio.PriorityQueue()
        self.jobs = {}
        self.attempts = {}
        self.cancelled = set()
        self.seq = 0

    def put(self, name, priority):
        self.seq += 1
        self.queue.put_nowait((priority, self.seq, name))

    def submit(self, name, priority, job):
        self.jobs[name] = (priority, job)
        self.attempts[name] = 0
        self.put(name, priority)

    def cancel(self, name):
        self.cancelled.add(name)

    async def worker(self, max_attempts):
        while True:
            priority, seq, name = await self.queue.get()
            try:
                if name in self.cancelled:
                    print("cancelled", name)
                    continue
                self.attempts[name] += 1
                attempt = self.attempts[name]
                job = self.jobs[name][1]
                try:
                    result = await job(attempt)
                except ConnectionError:
                    if attempt >= max_attempts:
                        print("failed", name)
                    else:
                        print(f"retry {name} (attempt {attempt})")
                        self.put(name, priority)
                else:
                    print(f"done {name}: {result}")
            finally:
                self.queue.task_done()

    async def run(self, max_attempts):
        worker = asyncio.create_task(self.worker(max_attempts))
        await self.queue.join()
        worker.cancel()


scheduler = Scheduler()
for name, priority, job in jobs:
    scheduler.submit(name, priority, job)
scheduler.cancel("cleanup")
await scheduler.run(max_attempts=2)
```

The queue orders tuples, so `(1, 2, 'backup')` comes before `(2, 3, 'email')`. `seq` breaks ties: `email` (seq 3) runs before `cleanup` (seq 4), and when `email` is retried it gets seq 6, so it goes *behind* `cleanup` but still ahead of `sync` (priority 3). Without `seq`, two equal priorities would fall back to comparing names (or, with job objects in the tuple, raise `TypeError`). `continue` inside `try` still runs the `finally`, so every `get()` is matched by exactly one `task_done()`, which is what lets `queue.join()` return. The worker loops forever, so `run` cancels it once the work is done (NB27).

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import itertools
from collections.abc import Awaitable, Callable
from dataclasses import dataclass, field

type Job = Callable[[int], Awaitable[str]]

@dataclass
class Scheduler:
    queue: asyncio.PriorityQueue[tuple[int, int, str]] = field(default_factory=asyncio.PriorityQueue)
    jobs: dict[str, tuple[int, Job]] = field(default_factory=dict)
    attempts: dict[str, int] = field(default_factory=dict)
    cancelled: set[str] = field(default_factory=set)
    _seq: itertools.count = field(default_factory=itertools.count)

    def _enqueue(self, name: str) -> None:
        self.queue.put_nowait((self.jobs[name][0], next(self._seq), name))

    def submit(self, name: str, priority: int, job: Job) -> None:
        self.jobs[name] = (priority, job)
        self.attempts[name] = 0
        self._enqueue(name)

    def cancel(self, name: str) -> None:
        self.cancelled.add(name)

    async def _execute(self, name: str, max_attempts: int) -> None:
        if name in self.cancelled:
            print("cancelled", name)
            return
        self.attempts[name] += 1
        attempt = self.attempts[name]
        try:
            result = await self.jobs[name][1](attempt)
        except ConnectionError:
            if attempt >= max_attempts:
                print("failed", name)
                return
            print(f"retry {name} (attempt {attempt})")
            self._enqueue(name)
        else:
            print(f"done {name}: {result}")

    async def _worker(self, max_attempts: int) -> None:
        while True:
            _, _, name = await self.queue.get()
            try:
                await self._execute(name, max_attempts)
            finally:
                self.queue.task_done()

    async def run(self, max_attempts: int, workers: int = 1) -> None:
        async with asyncio.TaskGroup() as tg:
            tasks = [tg.create_task(self._worker(max_attempts)) for _ in range(workers)]
            await self.queue.join()
            for task in tasks:
                task.cancel()


scheduler = Scheduler()
for name, priority, job in jobs:
    scheduler.submit(name, priority, job)
scheduler.cancel("cleanup")
await scheduler.run(max_attempts=2)
```

`itertools.count()` is the standard tie-breaker counter, and splitting `_execute` from `_worker` keeps the queue bookkeeping in one small loop. `TaskGroup` (NB27) makes sure no worker outlives `run`, even on errors. The `workers=` option shows the real trade-off: more workers give more throughput for I/O-bound jobs, but priority becomes a *hint* rather than a strict order, because a low-priority job already running is not interrupted. Cancellation here is *cooperative* (checked when a job is dequeued); cancelling a job that is already running needs `task.cancel()` on its own task.

</details>

In [ ]:
import asyncio

def flaky(fail_times):
    async def job(attempt):
        await asyncio.sleep(0)
        if attempt <= fail_times:
            raise ConnectionError("network down")
        return f"ok on attempt {attempt}"
    return job

# submit these in this order, then cancel "cleanup":
jobs = [("report", 5, flaky(0)), ("backup", 1, flaky(0)), ("email", 2, flaky(1)),
        ("cleanup", 2, flaky(0)), ("sync", 3, flaky(5))]

# your code here


### NB32-Q15 · L5 · Report pipeline bugs · Fix

A nightly report parses sales lines lazily, then applies threshold
alerts. It has **two** bugs: the average is wrong, and every alert uses
the same threshold. Fix both. Correct output:

```text
orders: 4 avg: 52.62
big(>40): ['A2', 'A3', 'A4']
big(>60): ['A3', 'A4']
big(>80): ['A4']
```

**Buggy code (copied into the cell below):**

```python
def parse(lines):
    for line in lines:
        order_id, amount = line.split(",")
        yield order_id, float(amount)

lines = ["A1,20", "A2,45", "A3,65", "A4,80.5"]
orders = parse(lines)
count = sum(1 for _ in orders)
total = sum(amount for _, amount in orders)
print("orders:", count, "avg:", round(total / count, 2) if count else 0)

alerts = []
for limit in [40, 60, 80]:
    alerts.append((limit, lambda amount: amount > limit))

for limit, is_big in alerts:
    print(f"big(>{limit}):", [oid for oid, amount in parse(lines) if is_big(amount)])
```

<details><summary><b>Hint</b></summary>

How many times can you loop over a generator? And when does a `lambda` look up the value of `limit`: when it is created, or when it is called?

</details>

<details><summary><b>Expected output</b></summary>

```text
orders: 4 avg: 52.62
big(>40): ['A2', 'A3', 'A4']
big(>60): ['A3', 'A4']
big(>80): ['A4']
```

</details>

<details><summary><b>Solution</b></summary>

```python
def parse(lines):
    for line in lines:
        order_id, amount = line.split(",")
        yield order_id, float(amount)

lines = ["A1,20", "A2,45", "A3,65", "A4,80.5"]
orders = list(parse(lines))
count = len(orders)
total = sum(amount for _, amount in orders)
print("orders:", count, "avg:", round(total / count, 2) if count else 0)

alerts = []
for limit in [40, 60, 80]:
    alerts.append((limit, lambda amount, limit=limit: amount > limit))

for limit, is_big in alerts:
    print(f"big(>{limit}):", [oid for oid, amount in parse(lines) if is_big(amount)])
```

Bug 1: a generator can be consumed only once. Counting used it up, so the second pass summed nothing and the average came out as `0.0`. Materialise it with `list()` when you need several passes (NB22). Bug 2: a closure looks up `limit` when it is *called*, and by then the loop has finished with `limit = 80`, so every alert used 80. The default argument `limit=limit` captures the value at creation time (NB10/NB23). Both bugs print plausible-looking numbers, which is what makes them dangerous.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from collections.abc import Callable, Iterable, Iterator
from functools import partial
from operator import lt

def parse(lines: Iterable[str]) -> Iterator[tuple[str, float]]:
    for line in lines:
        order_id, amount = line.split(",")
        yield order_id, float(amount)

lines = ["A1,20", "A2,45", "A3,65", "A4,80.5"]
count, total = 0, 0.0
for _, amount in parse(lines):
    count += 1
    total += amount
print("orders:", count, "avg:", round(total / count, 2) if count else 0)

alerts: list[tuple[int, Callable[[float], bool]]] = [
    (limit, partial(lt, limit)) for limit in [40, 60, 80]
]
for limit, is_big in alerts:
    print(f"big(>{limit}):", [oid for oid, amount in parse(lines) if is_big(amount)])
```

When the data might be huge, a senior keeps the stream lazy and computes every statistic in **one** pass instead of storing it all in a list: that is the memory-versus-simplicity trade-off (NB29). `partial(lt, limit)` builds `limit < amount` with the value bound immediately, so there is no late-binding closure to get wrong. The comprehension also gives each callable its own `limit` by construction.

</details>

In [ ]:
def parse(lines):
    for line in lines:
        order_id, amount = line.split(",")
        yield order_id, float(amount)

lines = ["A1,20", "A2,45", "A3,65", "A4,80.5"]
orders = parse(lines)
count = sum(1 for _ in orders)
total = sum(amount for _, amount in orders)
print("orders:", count, "avg:", round(total / count, 2) if count else 0)

alerts = []
for limit in [40, 60, 80]:
    alerts.append((limit, lambda amount: amount > limit))

for limit, is_big in alerts:
    print(f"big(>{limit}):", [oid for oid, amount in parse(lines) if is_big(amount)])


---

## Mini-project

Pull together everything from this notebook.

## NB32-P · L5 · Mini-project: Event-Sourced Bank Ledger · Write

In an *event-sourced* system you never store balances. You store an
append-only list of **events** (facts that happened), and any balance is
worked out by replaying them. That gives you a full audit trail and
"what was the balance at event 3?" for free. Build one.

**1. Events.** Four frozen dataclasses: `Opened(account, owner)`,
`Deposited(account, amount)`, `Withdrew(account, amount)` and
`Transferred(source, target, amount)`. Amounts are whole **cents**
(`int`), never floats.

**2. Folding.** `apply(state, event)` updates a dict of balances for one
event (`Opened` adds the account at `0`). `replay(events)` is a
**generator** that applies events one at a time and yields a *copy* of
the balances after each one.

**3. Ledger.** A class with `events`, `owners` and `balances`, and
command methods `open(account, owner)`, `deposit(account, amount)`,
`withdraw(account, amount)` and `transfer(source, target, amount)`.
Each command validates first and raises `LedgerError` (your own
exception class) with exactly these messages, or else records one
event:

- `unknown account Z-9` (for any account that was never opened),
- `amount must be positive`,
- `insufficient funds in B-2 (have 30.00, need 50.00)`.

**4. Atomic batches.** A context manager `atomic(ledger)`: if the block
raises, it drops every event recorded inside the block, rebuilds
`balances` by replaying the remaining events, and re-raises.

**5. Tests.** A `unittest.TestCase` with three tests: an overdraft is
rejected and records no event; a failed `atomic` batch leaves events
and balances unchanged; replaying all events gives the same balances as
the ledger. Run them with `unittest.TextTestRunner(stream=io.StringIO())`.

Run this scenario with the given `money()` helper:

```python
ledger = Ledger()
ledger.open("A-1", "Ada")
ledger.open("B-2", "Bo")
ledger.deposit("A-1", 10000)
ledger.transfer("A-1", "B-2", 3000)
ledger.withdraw("B-2", 5000)          # rejected
with atomic(ledger):                  # rolled back as a whole
    ledger.withdraw("A-1", 1000)
    ledger.transfer("A-1", "Z-9", 500)
```

catching each `LedgerError` and printing it as shown, then print the
report. Expected output:

```text
rejected: insufficient funds in B-2 (have 30.00, need 50.00)
batch rolled back: unknown account Z-9
== event log ==
1 Opened(account='A-1', owner='Ada')
2 Opened(account='B-2', owner='Bo')
3 Deposited(account='A-1', amount=10000)
4 Transferred(source='A-1', target='B-2', amount=3000)
== balances ==
A-1 Ada 70.00
B-2 Bo 30.00
== history of A-1 ==
1 Opened -> 0.00
3 Deposited -> 100.00
4 Transferred -> 70.00
== audit ==
replay matches: True
A-1 after event 3: 100.00
tests run: 3, ok: True
```

The history lists only events that involve `A-1`, with its balance
just after each one. Balances are printed in sorted account order.

**Starting code (already in the cell below):**

```python
import io
import unittest

def money(cents):
    return f"{cents // 100}.{cents % 100:02d}"
```

<details><summary><b>Hint</b></summary>

Build it bottom-up and test each layer: events, then `apply` with `match`, then `replay`, then the `Ledger` (record = append the event and `apply` it), then `atomic` with `@contextmanager` remembering `len(ledger.events)`. For the history, `zip(events, replay(events))` pairs each event with the balances after it.

</details>

<details><summary><b>Expected output</b></summary>

```text
rejected: insufficient funds in B-2 (have 30.00, need 50.00)
batch rolled back: unknown account Z-9
== event log ==
1 Opened(account='A-1', owner='Ada')
2 Opened(account='B-2', owner='Bo')
3 Deposited(account='A-1', amount=10000)
4 Transferred(source='A-1', target='B-2', amount=3000)
== balances ==
A-1 Ada 70.00
B-2 Bo 30.00
== history of A-1 ==
1 Opened -> 0.00
3 Deposited -> 100.00
4 Transferred -> 70.00
== audit ==
replay matches: True
A-1 after event 3: 100.00
tests run: 3, ok: True
```

</details>

<details><summary><b>Solution</b></summary>

```python
from contextlib import contextmanager
from dataclasses import dataclass


@dataclass(frozen=True)
class Opened:
    account: str
    owner: str

@dataclass(frozen=True)
class Deposited:
    account: str
    amount: int

@dataclass(frozen=True)
class Withdrew:
    account: str
    amount: int

@dataclass(frozen=True)
class Transferred:
    source: str
    target: str
    amount: int


class LedgerError(Exception):
    pass


def apply(state, event):
    match event:
        case Opened(account, _):
            state[account] = 0
        case Deposited(account, amount):
            state[account] += amount
        case Withdrew(account, amount):
            state[account] -= amount
        case Transferred(source, target, amount):
            state[source] -= amount
            state[target] += amount


def replay(events):
    state = {}
    for event in events:
        apply(state, event)
        yield dict(state)


def involves(event, account):
    match event:
        case Transferred(source, target, _):
            return account in (source, target)
        case _:
            return event.account == account


class Ledger:
    def __init__(self):
        self.events = []
        self.owners = {}
        self.balances = {}

    def record(self, event):
        self.events.append(event)
        apply(self.balances, event)
        if isinstance(event, Opened):
            self.owners[event.account] = event.owner

    def check_account(self, account):
        if account not in self.balances:
            raise LedgerError(f"unknown account {account}")

    def check_amount(self, amount):
        if amount <= 0:
            raise LedgerError("amount must be positive")

    def check_funds(self, account, amount):
        have = self.balances[account]
        if have < amount:
            raise LedgerError(f"insufficient funds in {account} "
                              f"(have {money(have)}, need {money(amount)})")

    def open(self, account, owner):
        self.record(Opened(account, owner))

    def deposit(self, account, amount):
        self.check_account(account)
        self.check_amount(amount)
        self.record(Deposited(account, amount))

    def withdraw(self, account, amount):
        self.check_account(account)
        self.check_amount(amount)
        self.check_funds(account, amount)
        self.record(Withdrew(account, amount))

    def transfer(self, source, target, amount):
        self.check_account(source)
        self.check_account(target)
        self.check_amount(amount)
        self.check_funds(source, amount)
        self.record(Transferred(source, target, amount))


@contextmanager
def atomic(ledger):
    mark = len(ledger.events)
    try:
        yield ledger
    except Exception:
        del ledger.events[mark:]
        ledger.balances = {}
        for event in ledger.events:
            apply(ledger.balances, event)
        raise


class LedgerTests(unittest.TestCase):
    def make_ledger(self):
        ledger = Ledger()
        ledger.open("A", "Ann")
        ledger.deposit("A", 500)
        return ledger

    def test_overdraft_rejected(self):
        ledger = self.make_ledger()
        with self.assertRaises(LedgerError):
            ledger.withdraw("A", 600)
        self.assertEqual(len(ledger.events), 2)

    def test_atomic_rolls_back(self):
        ledger = self.make_ledger()
        with self.assertRaises(LedgerError):
            with atomic(ledger):
                ledger.withdraw("A", 100)
                ledger.deposit("missing", 100)
        self.assertEqual(len(ledger.events), 2)
        self.assertEqual(ledger.balances, {"A": 500})

    def test_replay_matches(self):
        ledger = self.make_ledger()
        ledger.withdraw("A", 200)
        *_, final = replay(ledger.events)
        self.assertEqual(final, ledger.balances)


ledger = Ledger()
ledger.open("A-1", "Ada")
ledger.open("B-2", "Bo")
ledger.deposit("A-1", 10000)
ledger.transfer("A-1", "B-2", 3000)
try:
    ledger.withdraw("B-2", 5000)
except LedgerError as error:
    print("rejected:", error)
try:
    with atomic(ledger):
        ledger.withdraw("A-1", 1000)
        ledger.transfer("A-1", "Z-9", 500)
except LedgerError as error:
    print("batch rolled back:", error)

print("== event log ==")
for number, event in enumerate(ledger.events, start=1):
    print(number, event)

print("== balances ==")
for account in sorted(ledger.balances):
    print(account, ledger.owners[account], money(ledger.balances[account]))

print("== history of A-1 ==")
snapshots = replay(ledger.events)
for number, (event, state) in enumerate(zip(ledger.events, snapshots), start=1):
    if involves(event, "A-1"):
        print(number, type(event).__name__, "->", money(state["A-1"]))

print("== audit ==")
states = list(replay(ledger.events))
print("replay matches:", states[-1] == ledger.balances)
print("A-1 after event 3:", money(states[2]["A-1"]))

suite = unittest.defaultTestLoader.loadTestsFromTestCase(LedgerTests)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print(f"tests run: {result.testsRun}, ok: {result.wasSuccessful()}")
```

The core idea is that `balances` is a *cache* of `replay(events)`: the events are the truth. That makes rollback easy (cut the list back and replay) and audits easy (replay up to any point). Commands validate **before** recording, so an invalid command leaves no trace, and `atomic` covers the case where each command is valid alone but the batch fails half-way: the `withdraw` succeeded, the `transfer` to `Z-9` did not, and both vanish. Frozen dataclasses make events immutable facts, and `match` with class patterns keeps `apply` readable. Integer cents avoid every float rounding surprise from NB01 and NB16. `*_, final = replay(...)` is a neat way to take the last item of a generator.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from collections.abc import Iterable, Iterator
from contextlib import contextmanager
from dataclasses import dataclass
from itertools import islice


@dataclass(frozen=True, slots=True)
class Opened:
    account: str
    owner: str

@dataclass(frozen=True, slots=True)
class Deposited:
    account: str
    amount: int

@dataclass(frozen=True, slots=True)
class Withdrew:
    account: str
    amount: int

@dataclass(frozen=True, slots=True)
class Transferred:
    source: str
    target: str
    amount: int

type Event = Opened | Deposited | Withdrew | Transferred
type Balances = dict[str, int]


class LedgerError(Exception):
    """A command was rejected; nothing was recorded."""


def apply(state: Balances, event: Event) -> None:
    match event:
        case Opened(account, _):
            state[account] = 0
        case Deposited(account, amount):
            state[account] += amount
        case Withdrew(account, amount):
            state[account] -= amount
        case Transferred(source, target, amount):
            state[source] -= amount
            state[target] += amount


def replay(events: Iterable[Event]) -> Iterator[Balances]:
    state: Balances = {}
    for event in events:
        apply(state, event)
        yield state.copy()


def accounts_of(event: Event) -> tuple[str, ...]:
    match event:
        case Transferred(source, target, _):
            return (source, target)
        case Opened(account, _) | Deposited(account, _) | Withdrew(account, _):
            return (account,)


class Ledger:
    def __init__(self) -> None:
        self.events: list[Event] = []
        self.owners: dict[str, str] = {}
        self.balances: Balances = {}

    def _record(self, event: Event) -> None:
        self.events.append(event)
        apply(self.balances, event)
        if isinstance(event, Opened):
            self.owners[event.account] = event.owner

    def _require(self, *accounts: str, amount: int | None = None,
                 funds_from: str | None = None) -> None:
        for account in accounts:
            if account not in self.balances:
                raise LedgerError(f"unknown account {account}")
        if amount is not None and amount <= 0:
            raise LedgerError("amount must be positive")
        if funds_from is not None and (have := self.balances[funds_from]) < amount:
            raise LedgerError(f"insufficient funds in {funds_from} "
                              f"(have {money(have)}, need {money(amount)})")

    def open(self, account: str, owner: str) -> None:
        if account in self.balances:
            raise LedgerError(f"account {account} already exists")
        self._record(Opened(account, owner))

    def deposit(self, account: str, amount: int) -> None:
        self._require(account, amount=amount)
        self._record(Deposited(account, amount))

    def withdraw(self, account: str, amount: int) -> None:
        self._require(account, amount=amount, funds_from=account)
        self._record(Withdrew(account, amount))

    def transfer(self, source: str, target: str, amount: int) -> None:
        self._require(source, target, amount=amount, funds_from=source)
        self._record(Transferred(source, target, amount))

    def rebuild(self) -> None:
        self.balances = {}
        for event in self.events:
            apply(self.balances, event)


@contextmanager
def atomic(ledger: Ledger) -> Iterator[Ledger]:
    mark = len(ledger.events)
    try:
        yield ledger
    except BaseException:
        del ledger.events[mark:]
        ledger.rebuild()
        raise


class LedgerTests(unittest.TestCase):
    def setUp(self) -> None:
        self.ledger = Ledger()
        self.ledger.open("A", "Ann")
        self.ledger.deposit("A", 500)

    def test_overdraft_rejected(self) -> None:
        with self.assertRaises(LedgerError):
            self.ledger.withdraw("A", 600)
        self.assertEqual(len(self.ledger.events), 2)

    def test_atomic_rolls_back(self) -> None:
        with self.assertRaises(LedgerError), atomic(self.ledger):
            self.ledger.withdraw("A", 100)
            self.ledger.deposit("missing", 100)
        self.assertEqual(len(self.ledger.events), 2)
        self.assertEqual(self.ledger.balances, {"A": 500})

    def test_replay_matches(self) -> None:
        self.ledger.withdraw("A", 200)
        *_, final = replay(self.ledger.events)
        self.assertEqual(final, self.ledger.balances)


ledger = Ledger()
ledger.open("A-1", "Ada")
ledger.open("B-2", "Bo")
ledger.deposit("A-1", 10000)
ledger.transfer("A-1", "B-2", 3000)
try:
    ledger.withdraw("B-2", 5000)
except LedgerError as error:
    print("rejected:", error)
try:
    with atomic(ledger):
        ledger.withdraw("A-1", 1000)
        ledger.transfer("A-1", "Z-9", 500)
except LedgerError as error:
    print("batch rolled back:", error)

print("== event log ==")
for number, event in enumerate(ledger.events, start=1):
    print(number, event)

print("== balances ==")
for account, cents in sorted(ledger.balances.items()):
    print(account, ledger.owners[account], money(cents))

print("== history of A-1 ==")
timeline = enumerate(zip(ledger.events, replay(ledger.events)), start=1)
for number, (event, state) in timeline:
    if "A-1" in accounts_of(event):
        print(number, type(event).__name__, "->", money(state["A-1"]))

print("== audit ==")
*_, final = replay(ledger.events)
print("replay matches:", final == ledger.balances)
after_3 = next(islice(replay(ledger.events), 2, None))
print("A-1 after event 3:", money(after_3["A-1"]))

suite = unittest.defaultTestLoader.loadTestsFromTestCase(LedgerTests)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print(f"tests run: {result.testsRun}, ok: {result.wasSuccessful()}")
```

The senior version adds the `Event` union type (NB26) so a type checker flags an unhandled event in `apply`, uses `setUp` so each test gets a fresh ledger, and rejects opening an account twice. `islice` (NB22) reads the state after event 3 without building every snapshot in a list. The trade-offs worth naming: replaying from zero gets slow once there are millions of events, so real systems store periodic *snapshots* and replay only the tail; and events are forever, so changing an event's shape later needs versioning (an `upcaster` that converts old events). In exchange you get a perfect audit trail, which is why banks and accounting systems love this design.

</details>

In [ ]:
import io
import unittest

def money(cents):
    return f"{cents // 100}.{cents % 100:02d}"

# your code here


---

## Done

That's the whole course. Well played.